# Домашнє завдання: Внесення оновлень в БД і робота з транзакціями

Це ДЗ передбачене під виконання на локальній машині. Виконання з Google Colab буде суттєво ускладнене.

## Підготовка
1. Переконайтесь, що у вас встановлены необхідні бібліотеки:
   ```bash
   pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv
   ```

2. Створіть файл `.env` з параметрами підключення до бази даних classicmodels. Базу даних ви можете отримати через

  - docker-контейнер згідно існтрукції в [документі](https://www.notion.so/hannapylieva/Docker-1eb94835849480c9b2e7f5dc22ee4df9), також відео інструкції присутні на платформі - уроки "MySQL бази, клієнт для роботи з БД, Docker і ChatGPT для запитів" та "Як встановити Docker для роботи з базами даних без терміналу"
  - або встановивши локально цю БД - для цього перегляньте урок "Опціонально. Встановлення MySQL та  БД Сlassicmodels локально".
  
  Приклад `.env` файлу ми створювали в лекції. Ось його обовʼязкове наповнення:
    ```
    DB_HOST=your_host
    DB_PORT=3306 або 3307 - той, який Ви налаштували
    DB_USER=your_username
    DB_PASSWORD=your_password
    DB_NAME=classicmodels
    ```
  Якщо ви створили цей файл під час перегляду лекції - **новий створювати не треба**. Замініть лише назву БД, або пропишіть назву в коді створення підключення (замість отримання назви цільової БД зі змінних оточення). Але переконайтесь, що до `.env` файл лежить в тій самій папці, що і цей ноутбук.

  **УВАГА!** НЕ копіюйте скрит для **створення** `.env` файлу. В лекції він наводиться для прикладу. І давалось пояснення, що в реальних проєктах ми НІКОЛИ не пишемо доступи до бази в коді. Копіювання скрипта для створення `.env` файлу сюди в ДЗ буде вважатись грубою помилкою і ми зніматимемо бали.

3. Налаштуйте підключення через SQLAlchemy до БД за прикладом в лекції.

Рекомендую вивести (відобразити) змінну engine після створення. Вона має бути не None! Якщо None - значить у Вас не підтягнулись налаштування з .env файла.

Ви також можете налаштувати параметри підключення до БД без .env файла, просто прописавши текстом в відповідних місцях. Це - не рекомендований підхід.


## Завдання

### Завдання 1: Оновлення інформації про клієнта (2 бали)

**Створіть функцію для оновлення контактної інформації клієнта за його номером** з наступними можливостями:
- Оновлення телефону клієнта
- Оновлення email (якщо поле існує в таблиці)

Опціонально, якщо вам хочеться більше практики:
- Логування змін в окрему таблицю

Використайте підхід з параметризованими запитами через `text()` та `UPDATE` оператор. Не забудьте на початку перевірити чи існує клієнт з таким номером в базі - це хороша практика.

Отримати всі колонки, які існують в таблиці ви можете наступним запитом
```
  SELECT COLUMN_NAME, DATA_TYPE
  FROM INFORMATION_SCHEMA.COLUMNS
  WHERE TABLE_NAME = 'customers'
```

Запустіть функцію і продемонструйте її роботу, запустивши SELECT, який допоможе це зробити.



In [36]:
pip install sqlalchemy pymysql pandas matplotlib seaborn python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [38]:
import datetime
import requests
import json
import os

from dotenv import load_dotenv
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text, MetaData, Table
from sqlalchemy.orm import sessionmaker

In [39]:
load_dotenv(override=True)

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')

db = "classicmodels" 

connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    current_db = conn.execute(text("SELECT DATABASE()")).scalar()
    print(f"✅ Зараз підключено до бази: {current_db}")

✅ Зараз підключено до бази: classicmodels


In [40]:
data_chek = "SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = 'customers'"

df_data_chek = pd.read_sql(data_chek, engine)

display(df_data_chek)

,COLUMN_NAME,DATA_TYPE
0,customerNumber,int
1,customerName,varchar
2,contactLastName,varchar
3,contactFirstName,varchar
4,phone,varchar
5,addressLine1,varchar
6,addressLine2,varchar
7,city,varchar
8,state,varchar
9,postalCode,varchar


In [41]:
data_view = "SELECT customerNumber FROM customers LIMIT 5"

df_data_view = pd.read_sql(data_view, engine)

display(df_data_view)

,customerNumber
0,125
1,169
2,206
3,223
4,237


In [42]:
def update_customer_contact(engine, customer_number, new_phone, new_email):
    try:
        with engine.begin() as conn:
            conn.execute(text("""
                CREATE TABLE IF NOT EXISTS customer_log (
                    id INT AUTO_INCREMENT PRIMARY KEY,
                    customerNumber INT,
                    action VARCHAR(255),
                    log_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                )
            """))

            column_check = conn.execute(text("SHOW COLUMNS FROM customers")).fetchall()
            existing_columns = [col[0] for col in column_check]
            
            check_client = conn.execute(
                text("SELECT contactFirstName, contactLastName FROM customers WHERE customerNumber = :num"), 
                {'num': customer_number}
            ).fetchone()

            if not check_client:
                print(f"❌ Клієнт {customer_number} не знайдений.")
                return False

            print(f"👤 Клієнт: {check_client[0]} {check_client[1]} (ID: {customer_number})")

            sql_parts = ["phone = :phone"]
            params = {'phone': new_phone, 'num': customer_number}
            
            if 'email' in existing_columns:
                sql_parts.append("email = :email")
                params['email'] = new_email
            
            update_sql = text(f"UPDATE customers SET {', '.join(sql_parts)} WHERE customerNumber = :num")
            conn.execute(update_sql, params)

            conn.execute(
                text("INSERT INTO customer_log (customerNumber, action) VALUES (:num, :act)"),
                {'num': customer_number, 'act': f"Updated phone to {new_phone}"}
            )

            print(f"✅ Готово! Дані оновлено, запис у лог додано.")
            return True

    except Exception as e:
        print(f"❌ Помилка транзакції: {e}")
        return False

if update_customer_contact(engine, 125, "+380997778899", "new_boss@example.com"):
    with engine.connect() as conn:
        log_res = conn.execute(text("SELECT * FROM customer_log ORDER BY log_time DESC LIMIT 1")).fetchone()
        print(f"\n📝 Останній запис у лозі: {log_res}")

👤 Клієнт: Zbyszek  Piestrzeniewicz (ID: 125)
✅ Готово! Дані оновлено, запис у лог додано.

📝 Останній запис у лозі: (2, 125, 'Updated phone to +380997778899', datetime.datetime(2026, 3, 13, 9, 42, 53))


### Завдання 2: Створення нового замовлення з транзакцією (5 балів)

**Реалізуйте процес створення нового замовлення** з наступними кроками в одній транзакції:
- Створення запису в таблиці `orders`
- Додавання товарних позицій в `orderdetails`
- Перевірка наявності товарів на складі
- Зменшення кількості товарів на складі

Запустіть процес з тестовими даними і продемонструйте через SELECT, що процес успішно відпрацював і були виконані необхідні операції.




In [55]:
import pandas as pd
from IPython.display import display, HTML

tables = ['orders', 'orderdetails', 'products']

for table in tables:
    display(HTML(f"<h3>Аналіз таблиці: <span style='color: #2e76b2;'>{table}</span></h3>"))

    query_schema = f"SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '{table}'"
    df_schema = pd.read_sql(query_schema, engine)
    
    query_view = f"SELECT * FROM {table} LIMIT 5"
    df_view = pd.read_sql(query_view, engine)
    display(df_schema)
    display(df_view)

,COLUMN_NAME,DATA_TYPE
0,orderNumber,int
1,orderDate,date
2,requiredDate,date
3,shippedDate,date
4,status,varchar
5,comments,text
6,customerNumber,int


,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,None,363
1,10101,2003-01-09,2003-01-18,2003-01-11,Shipped,Check on availability.,128
2,10102,2003-01-10,2003-01-18,2003-01-14,Shipped,None,181
3,10103,2003-01-29,2003-02-07,2003-02-02,Shipped,None,121
4,10104,2003-01-31,2003-02-09,2003-02-01,Shipped,None,141


,COLUMN_NAME,DATA_TYPE
0,orderNumber,int
1,productCode,varchar
2,quantityOrdered,int
3,priceEach,decimal
4,orderLineNumber,smallint


,orderNumber,productCode,quantityOrdered,priceEach,orderLineNumber
0,10100,S18_1749,30,136.00,3
1,10100,S18_2248,50,55.09,2
2,10100,S18_4409,22,75.46,4
3,10100,S24_3969,49,35.29,1
4,10101,S18_2325,25,108.06,4


,COLUMN_NAME,DATA_TYPE
0,productCode,varchar
1,productName,varchar
2,productLine,varchar
3,productScale,varchar
4,productVendor,varchar
5,productDescription,text
6,quantityInStock,smallint
7,buyPrice,decimal
8,MSRP,decimal


,productCode,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,1:10,Min Lin Diecast,"This replica features working kickstand, front...",7933,48.81,95.70
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,1:10,Classic Metal Creations,Turnable front wheels; steering function; deta...,7305,98.58,214.30
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,1:10,Highway 66 Mini Classics,"Official Moto Guzzi logos and insignias, saddl...",6625,68.99,118.94
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,1:10,Red Start Diecast,"Model features, official Harley Davidson logos...",5582,91.02,193.66
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,1:10,Motor City Art Classics,Features include: Turnable front wheels; steer...,3252,85.68,136.00


In [59]:
from datetime import datetime

def create_full_order(engine, customer_id, items):
    try:
        with engine.begin() as conn:
            res = conn.execute(text("SELECT MAX(orderNumber) FROM orders"))
            new_order_number = (res.fetchone()[0] or 0) + 1

            conn.execute(text("""
                INSERT INTO orders (orderNumber, orderDate, requiredDate, status, customerNumber)
                VALUES (:num, :odate, :rdate, 'In Process', :cust)
            """), {
                'num': new_order_number,
                'odate': datetime.now().date(),
                'rdate': datetime.now().date(),
                'cust': customer_id
            })
            
            print(f"📦 Створено чернетку замовлення №{new_order_number}")

            for i, item in enumerate(items, 1):
                p_code = item['productCode']
                qty = item['quantity']
                
                product = conn.execute(
                    text("SELECT quantityInStock, productName FROM products WHERE productCode = :code"),
                    {'code': p_code}
                ).fetchone()
                
                if not product:
                    raise ValueError(f"Товар {p_code} не існує!")
                
                stock_available, p_name = product
                
                if stock_available < qty:
                    raise ValueError(f"Недостатньо '{p_name}'! Склад: {stock_available}, Треба: {qty}")

                conn.execute(text("""
                    INSERT INTO orderdetails (orderNumber, productCode, quantityOrdered, priceEach, orderLineNumber)
                    VALUES (:o_num, :p_code, :qty, :price, :line)
                """), {
                    'o_num': new_order_number,
                    'p_code': p_code,
                    'qty': qty,
                    'price': item['price'],
                    'line': i
                })

                conn.execute(text("""
                    UPDATE products 
                    SET quantityInStock = quantityInStock - :qty 
                    WHERE productCode = :p_code
                """), {'qty': qty, 'p_code': p_code})
                
                print(f"  ✅ {p_name}: додано {qty} шт. (Залишок: {stock_available - qty})")

            print(f"🎉 Замовлення №{new_order_number} успішно оформлено!")
            return new_order_number

    except Exception as e:
        print(f"❌ ТРАНЗАКЦІЮ СКАСОВАНО: {e}")
        return None

my_items = [
    {'productCode': 'S10_1678', 'quantity': 1, 'price': 95.70},
    {'productCode': 'S12_1108', 'quantity': 3, 'price': 207.35}
]

order_id = create_full_order(engine, 125, my_items)

if order_id:
    with engine.connect() as conn:
        print(f"\n--- РЕЗУЛЬТАТИ ДЛЯ ЗАМОВЛЕННЯ №{order_id} ---")

        o = conn.execute(text("SELECT orderNumber, status, customerNumber FROM orders WHERE orderNumber = :id"), {'id': order_id}).fetchone()
        print(f"Таблиця orders: {o}")
        
        od = conn.execute(text("SELECT productCode, quantityOrdered, priceEach FROM orderdetails WHERE orderNumber = :id"), {'id': order_id}).fetchall()
        print("Таблиця orderdetails:")
        for row in od:
            print(f"   - {row}")

        print("\nСклад оновлено успішно.")

📦 Створено чернетку замовлення №10427
  ✅ 1969 Harley Davidson Ultimate Chopper: додано 1 шт. (Залишок: 7931)
  ✅ 2001 Ferrari Enzo: додано 3 шт. (Залишок: 3613)
🎉 Замовлення №10427 успішно оформлено!

--- РЕЗУЛЬТАТИ ДЛЯ ЗАМОВЛЕННЯ №10427 ---
Таблиця orders: (10427, 'In Process', 125)
Таблиця orderdetails:
   - ('S10_1678', 1, Decimal('95.70'))
   - ('S12_1108', 3, Decimal('207.35'))

Склад оновлено успішно.
